# Cross-dataset validation - ULB credit card fraud

Everything so far was built and validated on PaySim, which is synthetic. This notebook checks whether the *methodology* carries over to a different dataset: the ULB / Kaggle credit card fraud data (real European card transactions, September 2013).

The trained PaySim model itself can't be reused here - the feature schemas have nothing in common (PaySim has engineered balance/destination features, ULB has anonymised PCA components V1-V28 plus Amount). So what's being tested is the approach:

- time-ordered split, never a random shuffle
- LightGBM, kept shallow, with the same tree settings as `02_modeling.ipynb`
- class imbalance handled by `scale_pos_weight`, **derived for this dataset** rather than copied from PaySim (copying another model's setting is what broke the first Random Forest comparison in `05_model_enhancement.ipynb`)
- PR-AUC with bootstrap 95% CIs, same `bootstrap_pr_auc` function as `02_modeling.ipynb`

Nothing here is tuned to match the PaySim numbers. The weight is chosen on a validation period and the held-out period is only used for the final report.

## Setup

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "creditcard.csv"
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(exist_ok=True)
RESULTS_PATH = REPORT_DIR / "06_cross_dataset_results.json"

# Same file as the Kaggle "Credit Card Fraud Detection" dataset (mlg-ulb/creditcardfraud).
# Downloaded from the public TensorFlow mirror because no Kaggle credentials are set up here.
DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"

SEED = 42

# Identical to LGB_PARAMS in 02_modeling.ipynb, except scale_pos_weight, which is derived below.
LGB_BASE_PARAMS = dict(
    n_estimators=200, num_leaves=15, max_depth=5, min_child_samples=100,
    reg_alpha=1.0, reg_lambda=1.0,
    random_state=SEED, verbose=-1,
)
PAYSIM_WEIGHT = 20   # what 02_modeling.ipynb used on PaySim - included in the sweep for reference only

print("lightgbm", lgb.__version__, "| sklearn", sklearn.__version__)

lightgbm 4.7.0 | sklearn 1.9.1


## Loading the data

In [2]:
if not DATA_PATH.exists():
    import urllib.request
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns.")
print(f"Fraud: {int(df['Class'].sum()):,} rows ({df['Class'].mean()*100:.4f}%)")
print(f"Time spans {df['Time'].min():.0f}s to {df['Time'].max():.0f}s ({df['Time'].max()/3600:.1f} hours)")
print(f"Missing values: {int(df.isna().sum().sum())} | exact duplicate rows: {int(df.duplicated().sum())}")

Loaded 284,807 rows, 31 columns.
Fraud: 492 rows (0.1727%)
Time spans 0s to 172792s (48.0 hours)


Missing values: 0 | exact duplicate rows: 1081


## Features + train / validation / held-out split

Split by time, same principle as PaySim. The data covers two days, so:

- **train** = day 1 (first 24h)
- **validation** = first half of day 2 (24h-36h) - used only to pick `scale_pos_weight`
- **held_out** = second half of day 2 (36h-48h) - only used for the final numbers

Features are V1-V28 and Amount. `Time` is left out: it's just seconds since the first transaction, and under a time-ordered split every held-out value lies outside the training range.

In [3]:
feature_columns = [f"V{i}" for i in range(1, 29)] + ["Amount"]
TARGET = "Class"

hours = df["Time"] / 3600
periods = {
    "train": df[hours < 24],
    "validation": df[(hours >= 24) & (hours < 36)],
    "held_out": df[hours >= 36],
}
X_train, y_train = periods["train"][feature_columns], periods["train"][TARGET]
X_val, y_val = periods["validation"][feature_columns], periods["validation"][TARGET]
X_held_out, y_held_out = periods["held_out"][feature_columns], periods["held_out"][TARGET]

print(f"{len(feature_columns)} features")
for name, part in periods.items():
    y = part[TARGET]
    print(f"  {name:11s} {len(part):>8,} rows  {int(y.sum()):>4,} fraud  ({y.mean()*100:.4f}%)")

n_legit, n_fraud = int((y_train == 0).sum()), int((y_train == 1).sum())
true_ratio = n_legit / n_fraud
print(f"\nTrain imbalance ratio (legit / fraud): {true_ratio:.1f}")

29 features
  train        144,786 rows   281 fraud  (0.1941%)
  validation    47,445 rows    96 fraud  (0.2023%)
  held_out      92,576 rows   115 fraud  (0.1242%)

Train imbalance ratio (legit / fraud): 514.3


## Helper - bootstrap PR-AUC
Same function as `02_modeling.ipynb`: predictions computed once, 200 resamples, 2.5th / 97.5th percentiles.

In [4]:
def bootstrap_pr_auc(model, X, y, n_bootstrap=200, seed=SEED):
    y_arr = np.asarray(y)
    preds = model.predict_proba(X)[:, 1]
    rng = np.random.RandomState(seed)
    n = len(y_arr)
    scores = []
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        if y_arr[idx].sum() == 0:  # PR-AUC undefined without positives
            continue
        scores.append(average_precision_score(y_arr[idx], preds[idx]))
    return np.array(scores)


def pr_auc_with_ci(model, X, y):
    point = average_precision_score(y, model.predict_proba(X)[:, 1])
    lo, hi = np.percentile(bootstrap_pr_auc(model, X, y), [2.5, 97.5])
    return float(point), float(lo), float(hi)

## Baseline
Logistic regression with balanced class weights, same sanity-check baseline as on PaySim.

In [5]:
baseline_model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train, y_train)
baseline_val = average_precision_score(y_val, baseline_model.predict_proba(X_val)[:, 1])
baseline_held = pr_auc_with_ci(baseline_model, X_held_out, y_held_out)
print(f"Baseline (LogReg) PR-AUC on validation: {baseline_val:.4f}")
print(f"Baseline (LogReg) PR-AUC on held_out:   {baseline_held[0]:.4f}   95% CI = [{baseline_held[1]:.4f}, {baseline_held[2]:.4f}]")

Baseline (LogReg) PR-AUC on validation: 0.8250
Baseline (LogReg) PR-AUC on held_out:   0.7365   95% CI = [0.6570, 0.8148]


## Deriving scale_pos_weight for this dataset

Sweeping the weight and scoring each on the **validation** period. The held-out column is printed next to it for transparency, but the choice is made on validation only - picking on held-out would make the final number optimistic.

The sweep includes PaySim's 20 and this dataset's true imbalance ratio.

In [6]:
weights = sorted({1, 5, 10, PAYSIM_WEIGHT, 50, 100, 200, round(true_ratio)})

sweep = []
models = {}
for w in weights:
    m = lgb.LGBMClassifier(**LGB_BASE_PARAMS, scale_pos_weight=w).fit(X_train, y_train)
    models[w] = m
    sweep.append({
        "scale_pos_weight": w,
        "train_pr_auc": average_precision_score(y_train, m.predict_proba(X_train)[:, 1]),
        "validation_pr_auc": average_precision_score(y_val, m.predict_proba(X_val)[:, 1]),
        "held_out_pr_auc": average_precision_score(y_held_out, m.predict_proba(X_held_out)[:, 1]),
    })
sweep_df = pd.DataFrame(sweep)
print(sweep_df.round(4).to_string(index=False))

best_weight = int(sweep_df.loc[sweep_df["validation_pr_auc"].idxmax(), "scale_pos_weight"])
print(f"\nChosen on validation: scale_pos_weight = {best_weight}")

 scale_pos_weight  train_pr_auc  validation_pr_auc  held_out_pr_auc
                1        1.0000             0.7553           0.6894
                5        1.0000             0.7642           0.6818
               10        0.9985             0.5872           0.5431
               20        1.0000             0.7557           0.6343
               50        0.9977             0.5435           0.4753
              100        0.9907             0.4527           0.2687
              200        1.0000             0.7000           0.5656
              514        0.2997             0.0928           0.0385

Chosen on validation: scale_pos_weight = 5


## Held-out result
The model with the validation-chosen weight, evaluated once on the held-out period.

In [7]:
final_model = models[best_weight]
train_pr_auc = average_precision_score(y_train, final_model.predict_proba(X_train)[:, 1])
val_result = pr_auc_with_ci(final_model, X_val, y_val)
held_result = pr_auc_with_ci(final_model, X_held_out, y_held_out)
paysim_weight_result = pr_auc_with_ci(models[PAYSIM_WEIGHT], X_held_out, y_held_out)

print(f"LightGBM (scale_pos_weight={best_weight})")
print(f"  train       PR-AUC = {train_pr_auc:.4f}   (gap vs held_out = overfitting check)")
print(f"  validation  PR-AUC = {val_result[0]:.4f}   95% CI = [{val_result[1]:.4f}, {val_result[2]:.4f}]")
print(f"  held_out    PR-AUC = {held_result[0]:.4f}   95% CI = [{held_result[1]:.4f}, {held_result[2]:.4f}]")
print(f"\nFor reference, PaySim's weight reused unchanged (scale_pos_weight={PAYSIM_WEIGHT}):")
print(f"  held_out    PR-AUC = {paysim_weight_result[0]:.4f}   95% CI = [{paysim_weight_result[1]:.4f}, {paysim_weight_result[2]:.4f}]")

LightGBM (scale_pos_weight=5)
  train       PR-AUC = 1.0000   (gap vs held_out = overfitting check)
  validation  PR-AUC = 0.7642   95% CI = [0.6720, 0.8522]
  held_out    PR-AUC = 0.6818   95% CI = [0.5947, 0.7662]

For reference, PaySim's weight reused unchanged (scale_pos_weight=20):
  held_out    PR-AUC = 0.6343   95% CI = [0.5549, 0.7202]


## Seed variance
Checked whether the held-out figure moves with the training seed. It doesn't: all five seeds give the identical score. `LGB_PARAMS` has no row or column subsampling, so `random_state` has nothing to randomise and training is deterministic. This check therefore says nothing about run-to-run variance for this configuration - the bootstrap CI is the only uncertainty estimate here.

In [8]:
seed_scores = []
for s in [42, 1, 2, 3, 4]:
    m = lgb.LGBMClassifier(**{**LGB_BASE_PARAMS, "random_state": s}, scale_pos_weight=best_weight).fit(X_train, y_train)
    seed_scores.append(average_precision_score(y_held_out, m.predict_proba(X_held_out)[:, 1]))
print("Held-out PR-AUC across seeds:", [f"{s:.4f}" for s in seed_scores])
print(f"mean = {np.mean(seed_scores):.4f}, std = {np.std(seed_scores):.4f}")

Held-out PR-AUC across seeds: ['0.6818', '0.6818', '0.6818', '0.6818', '0.6818']
mean = 0.6818, std = 0.0000


## Retraining on newer data
Same check as on PaySim: add the validation period to training and see whether held-out improves beyond the CI overlap.

In [9]:
X_retrain = pd.concat([X_train, X_val]); y_retrain = pd.concat([y_train, y_val])
retrained_model = lgb.LGBMClassifier(**LGB_BASE_PARAMS, scale_pos_weight=best_weight).fit(X_retrain, y_retrain)
retrained_result = pr_auc_with_ci(retrained_model, X_held_out, y_held_out)

print(f"Original  (day 1)        on held_out: {held_result[0]:.4f}  CI [{held_result[1]:.4f}, {held_result[2]:.4f}]")
print(f"Retrained (day 1 + val)  on held_out: {retrained_result[0]:.4f}  CI [{retrained_result[1]:.4f}, {retrained_result[2]:.4f}]")
print(f"Difference: {retrained_result[0] - held_result[0]:+.4f}")
overlap = not (retrained_result[2] < held_result[1] or held_result[2] < retrained_result[1])
print("CIs overlap -> no clear evidence either is better." if overlap else "CIs do not overlap.")

Original  (day 1)        on held_out: 0.6818  CI [0.5947, 0.7662]
Retrained (day 1 + val)  on held_out: 0.7022  CI [0.6125, 0.7923]
Difference: +0.0203
CIs overlap -> no clear evidence either is better.


## MLP on this dataset

`07_mlp_comparison.ipynb` found that a small MLP matched Random Forest and beat LightGBM on PaySim. Checking whether that also holds here, with the same network (hidden layers 64-32-16, dropout 0.2, weighted loss) and the same rules: class weight and early stopping chosen on the validation period, held-out used once.

Two differences from notebook 07, both because of the data: the batch size is 512 instead of 4096 (the training set is nine times smaller, so 4096 would leave only ~35 updates per epoch), and only `Amount` is log-transformed (V1-V28 are PCA components and just get standardised).

In [10]:
import torch
import torch.nn as nn

HIDDEN, DROPOUT, LEARNING_RATE, WEIGHT_DECAY = (64, 32, 16), 0.2, 1e-3, 1e-5
BATCH_SIZE, MAX_EPOCHS, PATIENCE = 512, 30, 5

def to_array(frame):
    values = frame[feature_columns].to_numpy(dtype=np.float64).copy()
    values[:, -1] = np.log1p(values[:, -1])   # Amount
    return values

train_values = to_array(X_train)
mlp_mean, mlp_std = train_values.mean(axis=0), train_values.std(axis=0)
to_tensor = lambda frame: torch.tensor((to_array(frame) - mlp_mean) / mlp_std, dtype=torch.float32)
T_train, T_val, T_held = to_tensor(X_train), to_tensor(X_val), to_tensor(X_held_out)
t_y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32)


def mlp_proba(model, T):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(T).squeeze(1)).numpy()


def train_mlp(pos_weight, seed=SEED):
    torch.manual_seed(seed)
    layers, size = [], T_train.shape[1]
    for hidden in HIDDEN:
        layers += [nn.Linear(size, hidden), nn.ReLU(), nn.Dropout(DROPOUT)]
        size = hidden
    model = nn.Sequential(*layers, nn.Linear(size, 1))
    optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(pos_weight)))
    generator = torch.Generator().manual_seed(seed)
    best_score, best_state, best_epoch, stale = -1.0, None, 0, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        order = torch.randperm(len(T_train), generator=generator)
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            optimiser.zero_grad()
            loss_fn(model(T_train[idx]).squeeze(1), t_y_train[idx]).backward()
            optimiser.step()
        score = average_precision_score(y_val, mlp_proba(model, T_val))
        if score > best_score:
            best_score, best_epoch, stale = score, epoch, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, best_score, best_epoch


mlp_sweep, mlp_models = [], {}
for w in [1, 5, 20, 50, 100, 500]:
    m, val_score, epoch = train_mlp(w)
    mlp_models[w] = m
    held_score = average_precision_score(y_held_out, mlp_proba(m, T_held))
    mlp_sweep.append({"pos_weight": w, "best_epoch": epoch, "validation_pr_auc": float(val_score), "held_out_pr_auc": float(held_score)})
    print(f"pos_weight={w:>4}: validation PR-AUC = {val_score:.4f} (epoch {epoch:2d})   held_out PR-AUC = {held_score:.4f}")
mlp_weight = max(mlp_sweep, key=lambda r: r["validation_pr_auc"])["pos_weight"]
print(f"\nChosen on validation: pos_weight = {mlp_weight}")

pos_weight=   1: validation PR-AUC = 0.8517 (epoch  4)   held_out PR-AUC = 0.7813


pos_weight=   5: validation PR-AUC = 0.8492 (epoch  4)   held_out PR-AUC = 0.7903


pos_weight=  20: validation PR-AUC = 0.8550 (epoch  4)   held_out PR-AUC = 0.7985


pos_weight=  50: validation PR-AUC = 0.8560 (epoch  4)   held_out PR-AUC = 0.7972


pos_weight= 100: validation PR-AUC = 0.8551 (epoch  4)   held_out PR-AUC = 0.7974


pos_weight= 500: validation PR-AUC = 0.8531 (epoch  4)   held_out PR-AUC = 0.8005

Chosen on validation: pos_weight = 50


In [11]:
class TorchWrapper:
    """Gives the MLP a predict_proba so the same bootstrap function can be used."""
    def __init__(self, model): self.model = model
    def predict_proba(self, T):
        p = mlp_proba(self.model, T)
        return np.column_stack([1 - p, p])

mlp_final = TorchWrapper(mlp_models[mlp_weight])
mlp_train_pr_auc = float(average_precision_score(y_train, mlp_proba(mlp_final.model, T_train)))
mlp_held = pr_auc_with_ci(mlp_final, T_held, y_held_out)
print(f"MLP (pos_weight={mlp_weight})")
print(f"  train     PR-AUC = {mlp_train_pr_auc:.4f}")
print(f"  held_out  PR-AUC = {mlp_held[0]:.4f}   95% CI = [{mlp_held[1]:.4f}, {mlp_held[2]:.4f}]")

mlp_seed_scores = [mlp_held[0]]
for s in [1, 2, 3, 4]:
    m, _, _ = train_mlp(mlp_weight, seed=s)
    mlp_seed_scores.append(float(average_precision_score(y_held_out, mlp_proba(m, T_held))))
print("\nHeld-out PR-AUC across seeds [42, 1, 2, 3, 4]:", [f"{s:.4f}" for s in mlp_seed_scores])
print(f"mean = {np.mean(mlp_seed_scores):.4f}, std = {np.std(mlp_seed_scores):.4f}")

MLP (pos_weight=50)
  train     PR-AUC = 0.7573
  held_out  PR-AUC = 0.7972   95% CI = [0.7312, 0.8613]



Held-out PR-AUC across seeds [42, 1, 2, 3, 4]: ['0.7972', '0.7949', '0.8030', '0.7962', '0.7915']
mean = 0.7965, std = 0.0037


## Saving the results
Written to `reports/06_cross_dataset_results.json` so the report and dashboard read the exact figures instead of retyping them.

In [12]:
results = {
    "dataset": "ULB / Kaggle credit card fraud (creditcard.csv)",
    "n_rows": int(len(df)), "n_fraud": int(df[TARGET].sum()), "fraud_rate": float(df[TARGET].mean()),
    "n_features": len(feature_columns),
    "split": {name: {"n_rows": int(len(p)), "n_fraud": int(p[TARGET].sum())} for name, p in periods.items()},
    "train_imbalance_ratio": float(true_ratio),
    "baseline_logreg": {"validation_pr_auc": float(baseline_val), "held_out_pr_auc": baseline_held[0],
                        "held_out_ci": [baseline_held[1], baseline_held[2]]},
    "weight_sweep": [{k: (int(v) if k == "scale_pos_weight" else float(v)) for k, v in row.items()} for row in sweep],
    "chosen_scale_pos_weight": best_weight,
    "lightgbm": {"train_pr_auc": float(train_pr_auc),
                 "validation_pr_auc": val_result[0], "validation_ci": [val_result[1], val_result[2]],
                 "held_out_pr_auc": held_result[0], "held_out_ci": [held_result[1], held_result[2]]},
    "lightgbm_paysim_weight": {"scale_pos_weight": PAYSIM_WEIGHT, "held_out_pr_auc": paysim_weight_result[0],
                               "held_out_ci": [paysim_weight_result[1], paysim_weight_result[2]]},
    "seed_variance": {"seeds": [42, 1, 2, 3, 4], "held_out_pr_auc": [float(s) for s in seed_scores],
                      "mean": float(np.mean(seed_scores)), "std": float(np.std(seed_scores))},
    "retrained": {"held_out_pr_auc": retrained_result[0], "held_out_ci": [retrained_result[1], retrained_result[2]]},
    "mlp": {"hidden_layers": list(HIDDEN), "dropout": DROPOUT, "batch_size": BATCH_SIZE,
            "pos_weight_sweep": mlp_sweep, "chosen_pos_weight": mlp_weight, "train_pr_auc": mlp_train_pr_auc,
            "held_out_pr_auc": mlp_held[0], "held_out_ci": [mlp_held[1], mlp_held[2]],
            "seed_variance": {"seeds": [42, 1, 2, 3, 4], "held_out_pr_auc": mlp_seed_scores,
                              "mean": float(np.mean(mlp_seed_scores)), "std": float(np.std(mlp_seed_scores))}},
    "n_bootstrap": 200, "seed": SEED,
}
RESULTS_PATH.write_text(json.dumps(results, indent=2))
print("Saved ->", RESULTS_PATH.relative_to(PROJECT_ROOT))

Saved -> reports\06_cross_dataset_results.json


## Findings

All numbers below are from the cells above (held-out = second half of day 2, 92,576 rows, 115 fraud).

| Model | Held-out PR-AUC | 95% CI |
|---|---|---|
| LightGBM, `scale_pos_weight=5` (chosen on validation) | 0.6818 | [0.5947, 0.7662] |
| LightGBM, `scale_pos_weight=20` (PaySim's value, reused unchanged) | 0.6343 | [0.5549, 0.7202] |
| LightGBM, `scale_pos_weight=5`, retrained on day 1 + validation | 0.7022 | [0.6125, 0.7923] |
| Logistic regression baseline | 0.7365 | [0.6570, 0.8148] |
| MLP (64-32-16, dropout 0.2, `pos_weight=50`) | 0.7972 | [0.7312, 0.8613] |

**What transferred**
- The pipeline ran on a different dataset without changes to the method: time-ordered split, same tree settings, weight derived on a validation period, bootstrap CIs.
- The derived weight (5) is different from PaySim's (20), and far from this dataset's true imbalance ratio (514). Using the true ratio gave a held-out PR-AUC of 0.0385.
- Retraining on more recent data made no clear difference (CIs overlap) - same conclusion as on PaySim.
- The MLP from `07_mlp_comparison.ipynb`, retrained on this dataset, has the highest held-out point estimate (0.7972; 0.7915 to 0.8030 across five seeds). Its CI overlaps both the baseline's and LightGBM's, so with 115 held-out fraud cases it cannot be separated from either. The ordering of the three point estimates (MLP, then the linear baseline, then LightGBM) is the same direction as the PaySim result that the MLP was ahead of LightGBM.

**What did not transfer**
- **LightGBM did not beat the logistic regression baseline here.** The baseline's point estimate is higher (0.7365 vs 0.6818); the CIs overlap, so the two are not statistically distinguishable, but there is no evidence of the advantage LightGBM had on PaySim (0.7113 vs 0.5676 on eval_1 in `02_modeling.ipynb`).
- **The PaySim regularisation settings do not control overfitting on this data.** Training PR-AUC is 1.0000 against 0.6818 held-out. On PaySim the same settings gave 0.8206 train vs 0.7198 held-out.
- **The weight sweep is unstable.** Validation PR-AUC does not change smoothly with the weight (for example 0.7642 at 5, 0.5872 at 10, 0.7557 at 20, 0.4527 at 100, 0.7000 at 200). The chosen weight is the best of an erratic set, not the top of a clear curve, and the difference between the derived weight and PaySim's 20 on held-out is within the CI overlap.

**Caveats**
- The held-out period has only 115 fraud rows, so the CI is wide (about +/-0.09). Small differences between rows of the table above should not be read as real.
- PR-AUC depends on the fraud base rate, which differs between the two datasets and between periods here (validation 0.202%, held-out 0.124%). The ULB and PaySim PR-AUC figures are not directly comparable as a measure of which dataset the model does "better" on.
- The ULB features are PCA components. Why the linear baseline holds up here was not investigated.
- Only the modelling and evaluation method was tested. The cost-based ALLOW / REVIEW / BLOCK thresholds, drift analysis and SHAP checks were not repeated on this dataset.